# Notebook 05 — Perfil Agregado por Parlamentar

**Sprint 3 — Lei e Política**

Perfil descritivo (não-supervisionado) de cada deputado por tema, consumido pelo frontend
da Sprint 4. Para cada par (parlamentar × tema) calculamos o percentual de votos
favoráveis e classificamos a postura.

## Pipeline

1. Junção `votos` × `votacoes` × `proposicoes` (tema_cidadao) × `parlamentares`
2. Agregação: `pct_favoravel` e `total_votacoes` por parlamentar × tema (mín. 3 votos decisivos)
3. Classificação de `postura_geral`: ≥65% → favorável · ≤35% → contrário · senão neutro
4. Gravação em `perfil_parlamentar`

In [1]:
import sys
sys.path.insert(0, '..')

import logging

import pandas as pd

from src.db import buscar_todos, upsert_perfil, get_client

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger('05_perfil')
print('Módulos carregados.')

Módulos carregados.


## 1. Montagem da base de votos

Cada linha é um voto decisivo (`favoravel`/`contrario`) de um deputado, ligado ao tema da
proposição. Juntamos `votos` → `votacoes` → `proposicoes` (`tema_cidadao`) →
`parlamentares`. Votos sem tema (votação sem proposição linkada) e abstenções saem do
cálculo de percentual.

In [2]:
# Carrega e junta as tabelas (escopo: Câmara — só há votos nominais da Câmara)
votos = pd.DataFrame(buscar_todos('votos', 'votacao_id,parlamentar_id,voto'))
votacoes = pd.DataFrame(buscar_todos('votacoes', 'id,proposicao_id'))
proposicoes = pd.DataFrame(buscar_todos('proposicoes', 'id,eixo,tema_cidadao'))
parlamentares = pd.DataFrame(buscar_todos('parlamentares', 'id,nome,casa'))

print(f'votos={len(votos)}  votacoes={len(votacoes)}  '
      f'proposicoes={len(proposicoes)}  parlamentares={len(parlamentares)}')

# Chaves de junção como inteiro anulável (proposicao_id vem como object por conter nulos)
for d, col in [(votos, 'votacao_id'), (votos, 'parlamentar_id'),
               (votacoes, 'id'), (votacoes, 'proposicao_id'),
               (proposicoes, 'id'), (parlamentares, 'id')]:
    d[col] = pd.to_numeric(d[col], errors='coerce').astype('Int64')

df = votos.merge(votacoes, left_on='votacao_id', right_on='id', suffixes=('', '_vt'))
df = df.merge(proposicoes, left_on='proposicao_id', right_on='id', suffixes=('', '_pr'))
df = df.merge(parlamentares, left_on='parlamentar_id', right_on='id', suffixes=('', '_pl'))

# Apenas Câmara, com tema definido e voto decisivo
df = df[df['casa'] == 'camara']
# So temas SUBSTANTIVOS (eixo B) viram perfil cidadao: formas A0/A1/A2
# (requerimentos, creditos) sao procedimentais e ficam fora do produto.
df = df[(df['tema_cidadao'].notna()) & (df['eixo'] == 'tema')]
df = df[df['voto'].isin(['favoravel', 'contrario'])].copy()

df['fav'] = (df['voto'] == 'favoravel').astype(int)

print(f'\nVotos decisivos com tema: {len(df)}')
print(f'Parlamentares distintos: {df["parlamentar_id"].nunique()}')
print(f'Temas distintos: {df["tema_cidadao"].nunique()}')

2026-06-26 19:29:20,298 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=0&limit=1000 "HTTP/2 200 OK"


2026-06-26 19:29:20,594 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=1000&limit=1000 "HTTP/2 200 OK"


2026-06-26 19:29:20,779 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=2000&limit=1000 "HTTP/2 200 OK"


2026-06-26 19:29:21,098 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=3000&limit=1000 "HTTP/2 200 OK"


2026-06-26 19:29:21,356 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=4000&limit=1000 "HTTP/2 200 OK"


...

2026-06-26 19:29:42,598 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ceixo%2Ctema_cidadao&offset=20000&limit=1000 "HTTP/2 200 OK"


2026-06-26 19:29:42,832 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ceixo%2Ctema_cidadao&offset=21000&limit=1000 "HTTP/2 200 OK"


2026-06-26 19:29:43,446 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ceixo%2Ctema_cidadao&offset=22000&limit=1000 "HTTP/2 200 OK"


2026-06-26 19:29:43,756 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/parlamentares?select=id%2Cnome%2Ccasa&offset=0&limit=1000 "HTTP/2 200 OK"


votos=55570  votacoes=137  proposicoes=22106  parlamentares=726

Votos decisivos com tema: 44112
Parlamentares distintos: 580
Temas distintos: 11


## 2. Agregação por parlamentar × tema

Para cada par (parlamentar, tema): `total_votacoes` = nº de votos decisivos e
`pct_favoravel` = % de favoráveis. Guardamos só pares com **≥ 3 votos** (evita
percentuais ruidosos de 1–2 votos). A postura segue os cortes do schema:
≥ 65% favorável · ≤ 35% contrário · senão neutro.

In [3]:
MIN_VOTOS = 1  # baixado de 3 para incluir temas com poucas votações (ex: Legislação Penal, Créditos)

agg = (
    df.groupby(['parlamentar_id', 'tema_cidadao'])
      .agg(total_votacoes=('fav', 'size'), favoraveis=('fav', 'sum'))
      .reset_index()
)
agg = agg[agg['total_votacoes'] >= MIN_VOTOS].copy()
agg['pct_favoravel'] = (agg['favoraveis'] / agg['total_votacoes'] * 100).round(2)

def classificar(pct):
    if pct >= 65:
        return 'favoravel'
    if pct <= 35:
        return 'contrario'
    return 'neutro'

agg['postura_geral'] = agg['pct_favoravel'].map(classificar)

print(f'Perfis (parlamentar × tema) com >= {MIN_VOTOS} votos: {len(agg)}')
print('\nDistribuição por postura_geral:')
print(agg['postura_geral'].value_counts())
print('\nTemas gerados:')
print(agg['tema_cidadao'].value_counts())
print('\nAmostra:')
print(agg.sort_values('total_votacoes', ascending=False).head(10).to_string(index=False))

Perfis (parlamentar × tema) com >= 1 votos: 5571

Distribuição por postura_geral:
postura_geral
favoravel    2534
neutro       1841
contrario    1196
Name: count, dtype: int64

Temas gerados:
tema_cidadao
Outras Políticas Públicas                   578
Tributação e Reforma Tributária             564
Criança e Adolescente                       520
Políticas de Prevenção e Incentivo          516
Administração e Serviços Públicos           514
Previdência e Assistência Social            510
Datas Comemorativas                         507
Segurança Pública                           504
Violência Doméstica e Direitos da Mulher    498
Direito Penal e Crimes                      489
Operações de Crédito de Municípios          371
Name: count, dtype: int64

Amostra:
 parlamentar_id              tema_cidadao  total_votacoes  favoraveis  pct_favoravel postura_geral
            311 Outras Políticas Públicas              65          35          53.85        neutro
            582 Outras Políticas 

## 3. Gravar em `perfil_parlamentar` e validar

`upsert_perfil` faz upsert por (`parlamentar_id`, `tema_cidadao`), então reexecutar o
notebook atualiza os perfis sem duplicar.

In [4]:
registros = [
    {
        'parlamentar_id': int(r['parlamentar_id']),
        'tema_cidadao': r['tema_cidadao'],
        'pct_favoravel': float(r['pct_favoravel']),
        'total_votacoes': int(r['total_votacoes']),
        'postura_geral': r['postura_geral'],
    }
    for _, r in agg.iterrows()
]
# Recalculo completo: limpa o perfil antes de gravar (evita temas antigos
# orfaos apos o redesign de clusterizacao). upsert_perfil so faz upsert por chave.
get_client().table('perfil_parlamentar').delete().neq('id', 0).execute()
total = upsert_perfil(registros)
print(f'{total} perfis gravados em perfil_parlamentar.')

2026-06-26 19:29:44,240 [INFO] HTTP Request: DELETE https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?id=neq.0 "HTTP/2 200 OK"


2026-06-26 19:29:44,753 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22total_votacoes%22%2C%22postura_geral%22%2C%22parlamentar_id%22%2C%22tema_cidadao%22%2C%22pct_favoravel%22 "HTTP/2 201 Created"


2026-06-26 19:29:45,219 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22total_votacoes%22%2C%22postura_geral%22%2C%22parlamentar_id%22%2C%22tema_cidadao%22%2C%22pct_favoravel%22 "HTTP/2 201 Created"


2026-06-26 19:29:45,644 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22total_votacoes%22%2C%22postura_geral%22%2C%22parlamentar_id%22%2C%22tema_cidadao%22%2C%22pct_favoravel%22 "HTTP/2 201 Created"


2026-06-26 19:29:46,124 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22total_votacoes%22%2C%22postura_geral%22%2C%22parlamentar_id%22%2C%22tema_cidadao%22%2C%22pct_favoravel%22 "HTTP/2 201 Created"


...

2026-06-26 19:29:50,166 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22total_votacoes%22%2C%22postura_geral%22%2C%22parlamentar_id%22%2C%22tema_cidadao%22%2C%22pct_favoravel%22 "HTTP/2 201 Created"


2026-06-26 19:29:51,892 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22total_votacoes%22%2C%22postura_geral%22%2C%22parlamentar_id%22%2C%22tema_cidadao%22%2C%22pct_favoravel%22 "HTTP/2 201 Created"


2026-06-26 19:29:52,653 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22total_votacoes%22%2C%22postura_geral%22%2C%22parlamentar_id%22%2C%22tema_cidadao%22%2C%22pct_favoravel%22 "HTTP/2 201 Created"


2026-06-26 19:29:52,662 [INFO] upsert perfil_parlamentar: 5571 registros


5571 perfis gravados em perfil_parlamentar.


In [5]:
# Sanidade: relê o que foi gravado
check = pd.DataFrame(buscar_todos(
    'perfil_parlamentar',
    'parlamentar_id,tema_cidadao,pct_favoravel,total_votacoes,postura_geral',
))
print(f'Perfis em perfil_parlamentar: {len(check)}')

if check.empty:
    print('\n⚠️  Tabela vazia. Causa provável: as proposições ligadas às votações ainda '
          'não têm `tema_cidadao` (rode o notebook 03 / Sprint 2 e grave os temas). '
          'Veja o print "Votos decisivos com tema" da célula 1: se for 0, é isso.')
else:
    print('\nDistribuição por postura_geral:')
    print(check['postura_geral'].value_counts())
    print('\nAmostra:')
    print(check.head(10).to_string(index=False))

2026-06-26 19:29:52,845 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?select=parlamentar_id%2Ctema_cidadao%2Cpct_favoravel%2Ctotal_votacoes%2Cpostura_geral&offset=0&limit=1000 "HTTP/2 200 OK"


2026-06-26 19:29:53,481 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?select=parlamentar_id%2Ctema_cidadao%2Cpct_favoravel%2Ctotal_votacoes%2Cpostura_geral&offset=1000&limit=1000 "HTTP/2 200 OK"


2026-06-26 19:29:54,189 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?select=parlamentar_id%2Ctema_cidadao%2Cpct_favoravel%2Ctotal_votacoes%2Cpostura_geral&offset=2000&limit=1000 "HTTP/2 200 OK"


2026-06-26 19:29:54,563 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?select=parlamentar_id%2Ctema_cidadao%2Cpct_favoravel%2Ctotal_votacoes%2Cpostura_geral&offset=3000&limit=1000 "HTTP/2 200 OK"


2026-06-26 19:29:54,964 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?select=parlamentar_id%2Ctema_cidadao%2Cpct_favoravel%2Ctotal_votacoes%2Cpostura_geral&offset=4000&limit=1000 "HTTP/2 200 OK"


2026-06-26 19:29:55,618 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?select=parlamentar_id%2Ctema_cidadao%2Cpct_favoravel%2Ctotal_votacoes%2Cpostura_geral&offset=5000&limit=1000 "HTTP/2 200 OK"


Perfis em perfil_parlamentar: 5571

Distribuição por postura_geral:
postura_geral
favoravel    2534
neutro       1841
contrario    1196
Name: count, dtype: int64

Amostra:
 parlamentar_id                             tema_cidadao  pct_favoravel  total_votacoes postura_geral
              6        Administração e Serviços Públicos          25.00               4     contrario
              6                    Criança e Adolescente          66.67               3     favoravel
              6                      Datas Comemorativas          50.00               6        neutro
              6                   Direito Penal e Crimes          75.00               4     favoravel
              6                Outras Políticas Públicas          68.00              50     favoravel
              6       Políticas de Prevenção e Incentivo          66.67               3     favoravel
              6         Previdência e Assistência Social          75.00               4     favoravel
            

In [6]:
print('votações c/ proposicao_id :', votacoes['proposicao_id'].notna().sum(), 'de', len(votacoes))
print('proposições c/ tema_cidadao:', proposicoes['tema_cidadao'].notna().sum(), 'de', len(proposicoes))

votações c/ proposicao_id : 137 de 137
proposições c/ tema_cidadao: 22106 de 22106
